# Notebook 18 — LSTM Sequence Model

**Goal:** Train an LSTM that reads the last 20 laps of a driver's race as a time series, and combine it with the 38 static tabular features and entity embeddings from NB15. Measure Spearman ρ vs MLP NB15 to test whether a sequence model adds architectural diversity to the Phase 3 super-ensemble.

## Why sequences instead of lag scalars

NB15 MLP flattens tyre trajectory into pre-engineered scalars: `LapTime_lag1`, `LapTime_lag2`, `LapTime_lag3`, rolling means, rolling stds, degradation slopes. These describe individual statistical properties of recent laps but cannot detect **trajectory shapes**. An LSTM operating on raw lap values can learn patterns like:
- "LapTime went 94.3 → 94.8 → 95.4 → 96.1 over laps 24–27" → exponential tyre cliff approaching
- "Position bounced 8 → 12 → 9 → 14 over 4 laps" → erratic handling on degraded rubber
- "TyreLife just reset to 1" → driver pitted last lap, this lap is stint-start (very low pit probability)

The key hypothesis is that **sequential ordering of raw values** carries signal the scalar summaries lose. This is an architectural diversity bet: if the LSTM and MLP fail on different laps, rank-averaging them raises the ensemble ceiling.

## Architecture

```
Sequence input:  (batch, 20, 6)  — 20-lap window × [LapTime, TyreLife, Cum_Deg, LapDelta, Position, PitStop]
    → LSTM(input=6, hidden=256, layers=2, dropout=0.2)
    → Last hidden state  →  (batch, 256)

Static input:    (batch, 38)  — all v2 tabular features (StandardScaled per fold)
    → Linear(38→128) + LayerNorm + ReLU  →  (batch, 128)

Entity embeddings:
    Driver(887→32) | Race(26→8) | Compound(5→3) | Year(4→2)  →  (batch, 45)

Merge: cat([256, 128, 45])  →  (batch, 429)
    → Linear(429→128) + BatchNorm + ReLU + Dropout(0.3)
    → Linear(128→1)   [logit]
```

Short windows (stint start, early race laps) are left-padded with zeros and a boolean mask is passed to `pack_padded_sequence` so the LSTM only processes valid steps.

## Training
- Loss: `BCEWithLogitsLoss(pos_weight=4.03)` — corrects the 5:1 negative:positive imbalance
- Optimizer: AdamW (lr=3e-4, wd=1e-4)
- Scheduler: CosineAnnealingLR (T_max=50) — warm decay over first 50 epochs, restarts if training continues
- Batch size: 2048 · Max epochs: 80 · Early stopping patience: 10
- Gradient clipping: norm=1.0 (prevents exploding gradients in LSTM layers)
- Hardware: Kaggle T4 GPU (~15–30 min total); CPU-only fallback works but is slow

## Phase 3 Ensemble Gate
For inclusion in the NB20 super-ensemble:
1. **Solo AUC ≥ 0.905** — minimum strength threshold (raised from 0.895 since MLP already hits 0.9102)
2. **Spearman ρ < 0.90 vs MLP NB15** — must bring genuinely different errors

**Inputs:** `train_features_v2.parquet`, `test_features_v2.parquet`, `fold_assignments.parquet`, `oof_predictions_nb15.parquet`  
**Outputs:** `oof_predictions_nb18.parquet` (lstm_oof), `test_predictions_nb18.parquet` (lstm_pred), `models/lstm_fold{0-4}.pt`, `models/nb18_summary.pkl`

In [ ]:
# nb18-01  Imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, rankdata
import pickle, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# nb18-02  Path detection + data load
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root = cwd
processed_dir = project_root / 'data' / 'processed'
models_dir    = project_root / 'models'
models_dir.mkdir(exist_ok=True)
print(f'Root: {project_root}')

train    = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test     = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds_df = pd.read_parquet(processed_dir / 'fold_assignments.parquet')
mlp_oof  = pd.read_parquet(processed_dir / 'oof_predictions_nb15.parquet')[['id', 'mlp_oof']]

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Folds: {folds_df.fold.value_counts().sort_index().to_dict()}')

## Feature Sets and Hyperparameters

This notebook uses **two distinct feature sets** with different roles:

**`FEAT_COLS` — 38 static tabular features (same v2 set as NB11–NB15)**  
Passed to the static FC branch. These are the engineered scalars: lag features, rolling stats, interaction terms, target encodings. The static branch gives the LSTM the same information the MLP has — the LSTM's job is to add on top of it, not replace it.

**`SEQ_COLS` — 6 raw columns for the 20-lap window**  
`[LapTime (s), TyreLife, Cumulative_Degradation_winsorized, LapTime_Delta, Position, PitStop]`

These are raw measurements, not engineered features. Using raw columns for the sequence is intentional: `LapTime_lag1/2/3` in FEAT_COLS already capture some recent history; if the LSTM window also used lagged scalars it would duplicate that information. Raw columns let the LSTM learn trajectory shapes the scalar summaries discard. `PitStop` (lagged by 1 via the window) tells the LSTM "driver just came out of the box" — laps with `TyreLife=1` in the window are safe stint-start indicators.

**`pos_weight = 4.03`** = (1 − 0.199) / 0.199 — upweights the minority class (pit laps) to prevent the loss from being dominated by the 80% of laps where PitNextLap=0.

In [ ]:
# nb18-03  Settings
FEAT_COLS = [
    'TyreLife_normalized_by_compound', 'TyreLife_sq', 'is_wet_tyre', 'compound_ordinal',
    'laps_remaining', 'is_testing_session', 'Stint', 'Position',
    'Degradation_rate', 'Degradation_acceleration', 'Cumulative_Degradation_winsorized',
    'LapTime_lag1', 'LapTime_lag2', 'LapTime_lag3',
    'LapTime_Delta_lag1', 'LapTime_Delta_lag2', 'LapTime_Delta_lag3',
    'LapTime_rolling_mean_3', 'LapTime_rolling_mean_5',
    'LapTime_rolling_std_3', 'LapTime_rolling_std_5',
    'Degradation_rolling_slope_3', 'Degradation_rolling_slope_5',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
    'TyreLife_x_compound_ordinal', 'Stint_x_compound_ordinal', 'TyreLife_x_cmpd_x_laps_rem',
    'prime_pit_window', 'prime_window_x_compound',
    'abs_position_change', 'pos_change_rolling_std_3', 'PitStop_lag1',
    'laps_to_driver_end', 'field_median_laptime', 'laptime_vs_field', 'field_pace_change',
]
assert len(FEAT_COLS) == 38, f'Expected 38, got {len(FEAT_COLS)}'

# Raw lap columns used as sequence window features
SEQ_COLS = ['LapTime (s)', 'TyreLife', 'Cumulative_Degradation_winsorized',
            'LapTime_Delta', 'Position', 'PitStop']

CAT_COLS   = ['Driver', 'Race', 'Compound', 'Year']
WINDOW     = 20         # laps of preceding history
POS_WEIGHT = 4.03       # (1-0.199)/0.199 for BCEWithLogitsLoss
BATCH_SIZE = 2048
MAX_EPOCHS = 80
PATIENCE   = 10
N_FOLDS    = 5

print(f'Static features: {len(FEAT_COLS)}, Seq features: {len(SEQ_COLS)}, Window: {WINDOW}')
if DEVICE.type == 'cpu':
    print('WARNING: running on CPU. Expect ~2-4 hrs for 5 folds. Upload to Kaggle T4 for 15-30 min.')

## Entity Embeddings

Categorical columns `Driver`, `Race`, `Compound`, `Year` are encoded as learned embedding vectors, identical to NB15. The label encoders are fitted on **combined train + test** so test-set inference never encounters an unknown index.

Embedding dimensions are chosen proportional to log(cardinality):
- `Driver` (887 unique) → dim 32 — largest category; each driver has a unique pit strategy style
- `Race` (26 unique) → dim 8 — each circuit has different pit-window timing
- `Compound` (5 unique) → dim 3 — tyre type affects expected stint length
- `Year` (4 unique) → dim 2 — captures year-on-year regulation changes

The embedding vocab size is set to `len(classes) + 1` to reserve index 0 for padding (used when the sequence window is left-padded with zeros — the driver/race/compound/year indices in those zero-padded steps would be 0, which maps to a dedicated padding embedding).

In [ ]:
# nb18-04  Label encoders for entity embeddings (fit on combined train+test)
combined_cats = pd.concat([train[CAT_COLS], test[CAT_COLS]], ignore_index=True)
label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(combined_cats[col].astype(str))
    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} classes')

for col in CAT_COLS:
    le = label_encoders[col]
    train[col + '_idx'] = le.transform(train[col].astype(str))
    test[col  + '_idx'] = le.transform(test[col].astype(str))

# Embedding (vocab_size+1, emb_dim) — +1 reserves index 0 for padding
EMB_DIMS = {
    'Driver':   (len(label_encoders['Driver'].classes_)   + 1, 32),
    'Race':     (len(label_encoders['Race'].classes_)     + 1, 8),
    'Compound': (len(label_encoders['Compound'].classes_) + 1, 3),
    'Year':     (len(label_encoders['Year'].classes_)     + 1, 2),
}
print('Embedding dims:', EMB_DIMS)

## Sequence Window Construction

For each row (lap), we build a fixed-length window of the **preceding 20 laps** from the same `(Race, Year, Driver)` group.

**Algorithm:**
1. Combine train and test, sort by `(Race, Year, Driver, LapNumber)` so each driver-race is a contiguous block in chronological order.
2. For row `i` within a group, look back `min(i, 20)` rows. Copy those lap values into the **right-aligned** end of the window array: `window[i, WINDOW - hist_len :]`.
3. Left-pad short windows with zeros (lap 1 has no history → the entire window is zero).
4. Set `mask[i, WINDOW - hist_len :]` = True for valid (non-padded) positions.

**Why the mask matters:**  
The LSTM processes zero-padded steps as if they were real laps, which would corrupt the hidden state with fake zero-data. `pack_padded_sequence` uses the mask's valid-step counts (`mask.sum(dim=1)`) to stop the LSTM at the last real lap, so only the actual history influences the hidden state.

**Scaling:**  
A `StandardScaler` is fitted on the **train split only** and applied to both train and test SEQ_COLS before window construction. This prevents test-set statistics from leaking into the scaler. Note that within the 5-fold CV loop, the static features (FEAT_COLS) get a separate per-fold scaler; the sequence scaler here is fit on all train rows once since it's used purely for the window values.

In [ ]:
# nb18-05  Build sequence windows
# Strategy: combine train+test, sort by (Race, Year, Driver, LapNumber) so each
# group is contiguous and in lap order. Build 20-lap preceding-history windows.
# Scale seq features before window building (fit scaler on train only).

idx_cols     = ['id', 'Race', 'Year', 'Driver', 'LapNumber']
cat_idx_cols = [c + '_idx' for c in CAT_COLS]

seq_scaler = StandardScaler()
seq_scaler.fit(train[SEQ_COLS].values)

# Deduplicate column list — Position and Cumulative_Degradation_winsorized
# appear in both SEQ_COLS and FEAT_COLS; avoid duplicate columns in DataFrame.
all_sel = list(dict.fromkeys(idx_cols + SEQ_COLS + FEAT_COLS + cat_idx_cols))

combined_df = pd.concat([
    train[all_sel + ['PitNextLap']].assign(is_train=True),
    test[ all_sel].assign(is_train=False, PitNextLap=-1),
], ignore_index=True)

combined_df = combined_df.sort_values(['Race', 'Year', 'Driver', 'LapNumber']).reset_index(drop=True)

# Normalise sequence features in place
combined_df[SEQ_COLS] = seq_scaler.transform(combined_df[SEQ_COLS].values).astype(np.float32)

N        = len(combined_df)
N_SF     = len(SEQ_COLS)
seq_vals = combined_df[SEQ_COLS].values.astype(np.float32)

all_windows = np.zeros((N, WINDOW, N_SF), dtype=np.float32)
all_masks   = np.zeros((N, WINDOW),       dtype=bool)

GROUP_COLS = ['Race', 'Year', 'Driver']
print(f'Building windows for {N:,} rows...')
t0 = time.time()

for _, grp_idx in tqdm(combined_df.groupby(GROUP_COLS, sort=False).groups.items(),
                       total=combined_df[GROUP_COLS].drop_duplicates().shape[0]):
    idxs  = grp_idx.values          # row positions in combined_df (in LapNumber order)
    n_grp = len(idxs)
    for i in range(n_grp):
        hist_len = min(i, WINDOW)
        if hist_len > 0:
            all_windows[idxs[i], WINDOW - hist_len:] = seq_vals[idxs[i - hist_len : i]]
            all_masks[idxs[i],   WINDOW - hist_len:] = True

print(f'Done in {time.time()-t0:.1f}s')
print(f'Rows with full history (>=20 prior laps): {all_masks.all(axis=1).sum():,}')
print(f'Rows with zero history (race/stint start): {(~all_masks.any(axis=1)).sum():,}')

In [ ]:
# nb18-06  Split back to train / test arrays
tr_mask = combined_df['is_train'].values
te_mask = ~tr_mask

train_windows    = all_windows[tr_mask]
train_seq_masks  = all_masks[tr_mask]
test_windows     = all_windows[te_mask]
test_seq_masks   = all_masks[te_mask]

train_static_raw = combined_df.loc[tr_mask, FEAT_COLS].values.astype(np.float32)
test_static_raw  = combined_df.loc[te_mask, FEAT_COLS].values.astype(np.float32)

train_targets = combined_df.loc[tr_mask, 'PitNextLap'].values.astype(np.float32)
train_ids     = combined_df.loc[tr_mask, 'id'].values
test_ids      = combined_df.loc[te_mask, 'id'].values

# Align fold assignments to the (now sorted) train order
fold_lookup  = folds_df.set_index('id')['fold']
train_folds  = fold_lookup.loc[train_ids].values

train_cat = {c: combined_df.loc[tr_mask, c+'_idx'].values for c in CAT_COLS}
test_cat  = {c: combined_df.loc[te_mask, c+'_idx'].values for c in CAT_COLS}

del all_windows, all_masks, combined_df  # free memory

print(f'Train windows: {train_windows.shape},  static: {train_static_raw.shape}')
print(f'Test  windows: {test_windows.shape},  static: {test_static_raw.shape}')
print(f'Fold counts: {pd.Series(train_folds).value_counts().sort_index().to_dict()}')

## Model Architecture

Three parallel branches merge into a single classification head:

```
SEQUENCE BRANCH
  window: (B, 20, 6)  ← 20-lap history of [LapTime, TyreLife, Cum_Deg, LapDelta, Pos, PitStop]
  mask:   (B, 20)     ← True = valid lap, False = left-padded zero
      ↓  pack_padded_sequence (skip padded steps)
  LSTM(input=6, hidden=256, layers=2, dropout=0.2)
      ↓  take last-layer final hidden state h_n[-1]
  lstm_feat:  (B, 256)

STATIC BRANCH
  static: (B, 38)  ← all 38 v2 tabular features (StandardScaled per CV fold)
      ↓  Linear(38→128) + LayerNorm + ReLU
  static_feat: (B, 128)

EMBEDDING BRANCH
  Driver(887→32) | Race(26→8) | Compound(5→3) | Year(4→2)
      ↓  concatenate
  emb: (B, 45)

MERGE
  cat([lstm_feat, static_feat, emb])  →  (B, 429)
      ↓  Linear(429→128) + BatchNorm1d + ReLU + Dropout(0.3)
      ↓  Linear(128→1)   ← raw logit
```

**Key design choices:**
- `pack_padded_sequence` with `enforce_sorted=False` handles variable-length histories without needing sorted batches. The LSTM's hidden state at the end of a padded sequence equals its state after the last valid lap.
- `LayerNorm` on the static branch (not BatchNorm) because static features have very different scales and the per-feature normalization is more stable here.
- `BatchNorm1d` in the head works well because all three branches have been projected to controlled-scale representations by the time they merge.
- Gradient clipping at norm=1.0 in `run_epoch` prevents the LSTM's recurrent connections from producing exploding gradients on long valid sequences.

In [ ]:
# nb18-07  Model
class F1LSTMModel(nn.Module):
    """LSTM on 20-lap window + static 38-feature branch + entity embeddings."""

    def __init__(self, n_static=38, n_seq=6, hidden=256, n_layers=2, lstm_drop=0.2,
                 emb_dims=None):
        super().__init__()
        if emb_dims is None:
            emb_dims = EMB_DIMS

        self.driver_emb   = nn.Embedding(*emb_dims['Driver'])
        self.race_emb     = nn.Embedding(*emb_dims['Race'])
        self.compound_emb = nn.Embedding(*emb_dims['Compound'])
        self.year_emb     = nn.Embedding(*emb_dims['Year'])
        emb_total = sum(v[1] for v in emb_dims.values())  # 32+8+3+2=45

        self.lstm = nn.LSTM(
            input_size=n_seq,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            dropout=lstm_drop if n_layers > 1 else 0.0,
        )

        self.static_fc = nn.Sequential(
            nn.Linear(n_static, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
        )

        merge = hidden + 128 + emb_total  # 256+128+45=429
        self.head = nn.Sequential(
            nn.Linear(merge, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )

    def forward(self, window, mask, static, driver, race, compound, year):
        emb = torch.cat([
            self.driver_emb(driver),
            self.race_emb(race),
            self.compound_emb(compound),
            self.year_emb(year),
        ], dim=1)  # (B, 45)

        # Pack sequence to skip padded steps
        seq_len = mask.sum(dim=1).long().clamp(min=1)
        packed  = nn.utils.rnn.pack_padded_sequence(
            window, seq_len.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        lstm_feat = h_n[-1]  # last layer hidden: (B, hidden)

        static_feat = self.static_fc(static)  # (B, 128)

        merged = torch.cat([lstm_feat, static_feat, emb], dim=1)  # (B, 429)
        return self.head(merged).squeeze(1)  # (B,)


# Sanity-check param count
_m = F1LSTMModel()
n_params = sum(p.numel() for p in _m.parameters())
print(f'Model params: {n_params:,}')
del _m

In [ ]:
# nb18-08  Dataset + DataLoader
class F1SeqDataset(Dataset):
    def __init__(self, windows, masks, static, cat_idxs, targets=None):
        self.windows  = torch.tensor(windows, dtype=torch.float32)
        self.masks    = torch.tensor(masks,   dtype=torch.bool)
        self.static   = torch.tensor(static,  dtype=torch.float32)
        self.driver   = torch.tensor(cat_idxs['Driver'],   dtype=torch.long)
        self.race     = torch.tensor(cat_idxs['Race'],     dtype=torch.long)
        self.compound = torch.tensor(cat_idxs['Compound'], dtype=torch.long)
        self.year     = torch.tensor(cat_idxs['Year'],     dtype=torch.long)
        self.targets  = (None if targets is None
                         else torch.tensor(targets, dtype=torch.float32))

    def __len__(self):  return len(self.windows)

    def __getitem__(self, i):
        d = dict(window=self.windows[i], mask=self.masks[i], static=self.static[i],
                 driver=self.driver[i], race=self.race[i],
                 compound=self.compound[i], year=self.year[i])
        if self.targets is not None:
            d['target'] = self.targets[i]
        return d


def make_loader(windows, masks, static, cat_idxs, targets=None, shuffle=False):
    ds = F1SeqDataset(windows, masks, static, cat_idxs, targets)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

In [ ]:
# nb18-09  Training utilities
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, all_logits = 0.0, []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            win  = batch['window'].to(DEVICE)
            mask = batch['mask'].to(DEVICE)
            stat = batch['static'].to(DEVICE)
            drv  = batch['driver'].to(DEVICE)
            rc   = batch['race'].to(DEVICE)
            cmp  = batch['compound'].to(DEVICE)
            yr   = batch['year'].to(DEVICE)

            logits = model(win, mask, stat, drv, rc, cmp, yr)

            if is_train:
                tgt  = batch['target'].to(DEVICE)
                loss = criterion(logits, tgt)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item() * len(win)

            all_logits.append(logits.detach().cpu().float().numpy())

    probs = torch.sigmoid(torch.tensor(np.concatenate(all_logits))).numpy()
    return probs, total_loss / max(len(loader.dataset), 1)

## 5-Fold GroupKFold Cross-Validation

Same GroupKFold by `Race + '_' + Year` as all other notebooks. For each fold:

1. **Scale static features** — fit a fresh `StandardScaler` on the training rows of that fold only, then transform both validation and test rows with it. This prevents any leakage of validation feature statistics into the model.
2. **Build loaders** — training loader shuffles to break temporal batch ordering; validation and test loaders do not shuffle.
3. **Train** — AdamW with CosineAnnealingLR, early stopping when val AUC fails to improve for 10 consecutive epochs. Best model weights are saved in CPU memory and restored before generating OOF predictions.
4. **OOF predictions** — generated from the best-epoch model. These are used for the correlation analysis and ensemble evaluation in later cells.
5. **Test predictions** — generated once per fold with the best model. The final test predictions are the mean across all 5 folds.
6. **Save fold model** — `models/lstm_fold{k}.pt` stores the best-epoch state dict for reproducibility.

In [ ]:
# nb18-10  5-fold CV training
oof_preds          = np.zeros(len(train_ids))
test_preds_folds   = []
fold_aucs          = []

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE)
)

for fold in range(N_FOLDS):
    print(f'\n{"="*60}')
    print(f'Fold {fold}')
    print('='*60)

    tr_idx = np.where(train_folds != fold)[0]
    va_idx = np.where(train_folds == fold)[0]

    # Scale static features (fit on train fold only)
    scaler   = StandardScaler()
    tr_stat  = scaler.fit_transform(train_static_raw[tr_idx])
    va_stat  = scaler.transform(train_static_raw[va_idx])
    te_stat  = scaler.transform(test_static_raw)

    tr_cat = {c: train_cat[c][tr_idx] for c in CAT_COLS}
    va_cat = {c: train_cat[c][va_idx] for c in CAT_COLS}

    tr_loader = make_loader(train_windows[tr_idx], train_seq_masks[tr_idx],
                            tr_stat, tr_cat,
                            targets=train_targets[tr_idx], shuffle=True)
    va_loader = make_loader(train_windows[va_idx], train_seq_masks[va_idx],
                            va_stat, va_cat)
    te_loader = make_loader(test_windows, test_seq_masks, te_stat, test_cat)

    model     = F1LSTMModel().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    best_auc, best_epoch, patience_ctr, best_state = 0.0, 0, 0, None

    for epoch in range(MAX_EPOCHS):
        _, tr_loss  = run_epoch(model, tr_loader, criterion, optimizer)
        scheduler.step()
        val_probs, _ = run_epoch(model, va_loader, criterion)
        val_auc      = roc_auc_score(train_targets[va_idx], val_probs)

        if val_auc > best_auc:
            best_auc   = val_auc
            best_epoch = epoch
            patience_ctr = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1

        if (epoch + 1) % 10 == 0 or patience_ctr >= PATIENCE:
            print(f'  ep {epoch+1:3d}  loss={tr_loss:.4f}  val={val_auc:.4f}  '
                  f'best={best_auc:.4f} (ep{best_epoch+1})')

        if patience_ctr >= PATIENCE:
            print(f'  Early stop')
            break

    # Restore best
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    oof_probs, _ = run_epoch(model, va_loader, criterion)
    oof_preds[va_idx] = oof_probs
    fold_auc = roc_auc_score(train_targets[va_idx], oof_probs)
    fold_aucs.append(fold_auc)
    print(f'  Fold {fold} AUC: {fold_auc:.4f}')

    te_probs, _ = run_epoch(model, te_loader, criterion)
    test_preds_folds.append(te_probs)

    torch.save(best_state, models_dir / f'lstm_fold{fold}.pt')
    print(f'  Saved lstm_fold{fold}.pt  (best ep {best_epoch+1})')

oof_auc = roc_auc_score(train_targets, oof_preds)
print(f'\nOOF AUC : {oof_auc:.4f}')
print(f'Fold AUCs: {[f"{a:.4f}" for a in fold_aucs]}')
print(f'Fold std : {np.std(fold_aucs):.4f}')

## Correlation Analysis and Phase 3 Gate

The diversity check is the most important diagnostic for Phase 3. A high solo AUC means nothing for the ensemble if the model fails on the same laps as MLP.

**Spearman ρ** measures rank correlation between the LSTM's OOF probabilities and the MLP's (NB15). A value close to 1 means both models assign similar relative scores to every lap — adding them to a rank average yields minimal gain because the rank ordering barely changes. A value well below 0.90 means they fail on structurally different laps, and combining them can push past either model's ceiling.

**Gate thresholds (Phase 3):**
- Solo AUC ≥ 0.905 — minimum useful solo performance (raised from 0.895 since MLP already scores 0.9102)
- ρ < 0.90 vs MLP — diversity sufficient for ensemble benefit

Both gates must pass for LSTM to be included in the NB20 super-ensemble. A model that passes AUC but fails ρ should still have its outputs saved — NB20 will re-examine the full ρ matrix and may include it if it brings diversity relative to LGBM even if it's correlated with MLP.

In [ ]:
# nb18-11  Correlation analysis + gate check
mlp_vals = mlp_oof.set_index('id').loc[train_ids, 'mlp_oof'].values

rho, _   = spearmanr(oof_preds, mlp_vals)
mlp_auc  = roc_auc_score(train_targets, mlp_vals)

print(f'LSTM solo OOF AUC : {oof_auc:.4f}')
print(f'MLP  solo OOF AUC : {mlp_auc:.4f}')
print(f'Spearman rho (LSTM vs MLP): {rho:.4f}')
print()
print(f'Gate: solo AUC >= 0.905  -> {"PASS" if oof_auc >= 0.905 else "FAIL"}')
print(f'Gate: rho  <  0.90 vs MLP -> {"PASS" if rho < 0.90 else "FAIL"}')

# Rank-average LSTM + MLP preview
lstm_rank = rankdata(oof_preds) / len(oof_preds)
mlp_rank  = rankdata(mlp_vals)  / len(mlp_vals)
ens_auc   = roc_auc_score(train_targets, (lstm_rank + mlp_rank) / 2)
print(f'\nRank avg LSTM+MLP OOF AUC: {ens_auc:.4f}  (delta vs MLP: {ens_auc-mlp_auc:+.4f})')

In [ ]:
# nb18-12  Average test predictions across folds
test_preds_mean = np.mean(np.stack(test_preds_folds, axis=0), axis=0)
print(f'Test preds: shape={test_preds_mean.shape}, mean={test_preds_mean.mean():.4f}, '
      f'std={test_preds_mean.std():.4f}')

In [ ]:
# nb18-13  Save outputs
oof_df = pd.DataFrame({
    'id':         train_ids,
    'fold':       train_folds,
    'PitNextLap': train_targets.astype(int),
    'lstm_oof':   oof_preds,
})
assert len(oof_df) == 439140, f'Expected 439140 rows, got {len(oof_df)}'
assert oof_df['id'].nunique() == 439140, 'Duplicate ids in OOF'
oof_df.to_parquet(processed_dir / 'oof_predictions_nb18.parquet', index=False)
print(f'Saved oof_predictions_nb18.parquet  ({len(oof_df):,} rows)')

test_out = pd.DataFrame({'id': test_ids, 'lstm_pred': test_preds_mean})
assert len(test_out) == len(test_ids), 'Row count mismatch in test predictions'
test_out.to_parquet(processed_dir / 'test_predictions_nb18.parquet', index=False)
print(f'Saved test_predictions_nb18.parquet ({len(test_out):,} rows)')

summary = {
    'oof_auc':              oof_auc,
    'fold_aucs':            fold_aucs,
    'fold_std':             float(np.std(fold_aucs)),
    'rho_vs_mlp':           float(rho),
    'ensemble_auc_lstm_mlp': float(ens_auc),
    'model':                'LSTM-NB18',
    'architecture':         'LSTM(6,256,2L)+Static(38->128)+Emb(45) -> head(429->128->1)',
    'window_size':          WINDOW,
    'gate_pass':            bool(oof_auc >= 0.905 and rho < 0.90),
}
with open(models_dir / 'nb18_summary.pkl', 'wb') as f:
    pickle.dump(summary, f)

print(f'\n=== NB18 Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

## Summary

**Artifacts written:**
- `data/processed/oof_predictions_nb18.parquet` — columns: `id`, `fold`, `PitNextLap`, `lstm_oof`
- `data/processed/test_predictions_nb18.parquet` — columns: `id`, `lstm_pred` (5-fold averaged)
- `models/lstm_fold{0-4}.pt` — best-epoch PyTorch weights per fold
- `models/nb18_summary.pkl` — AUCs, ρ vs MLP, gate result, architecture string

**NB20 super-ensemble inputs (from this notebook):**
- `oof_predictions_nb18.parquet` → `lstm_oof` column
- `test_predictions_nb18.parquet` → `lstm_pred` column

**Gate result interpretation:**
- If both gates pass → LSTM is included in NB20 rank average alongside MLP, LGBM, TFT, Hybrid.
- If AUC passes but ρ fails → outputs are still saved. NB20 will evaluate a 3-model LGBM+MLP+LSTM ensemble using the full ρ matrix. If ρ(LSTM, LGBM) is lower than ρ(LSTM, MLP), LSTM may still contribute net diversity through the LGBM pairing.
- If AUC fails → LSTM excluded; current best ensemble remains MLP+LGBM (v004, LB 0.93160).

**Next notebook:** NB19 — Simplified TFT Encoder. Uses Variable Selection Networks (VSN) to learn per-feature importance at each timestep, LSTM encoder initialized from static context, and temporal self-attention. Target: ρ ~0.78–0.83 vs MLP (lower than LSTM's ρ because VSN feature routing produces structurally different errors).